# Organize AMG benchmark datasets

CheckAMG annotate, as well as DRAM-V and VIBRANT, will be run on (1) metagenomes, (2) viromes, and (3) complete virus genomes, from multiple environments (soil, aquatic, and human gut).

In [2]:
! pip install polars pyfastatools --quiet

## Dataset sources
* Metagenomes and viromes
    * Using (a subset of) the same metagenomes and viromes in [Kosmopoulos et al. (2024) *Microbiome*](https://doi.org/10.1186/s40168-024-01905-x), which are paired viromes and metagenomes from:
        * Human fecal samples ([Shkoporov et al. (2019) *Cell Host Microbe*](https://doi.org/10.1016/j.chom.2019.09.009))
        * Freshwater samples from Lake Mendota ([Tran et al. (2023) *bioRxiv*](https://doi.org/10.1101/2023.04.19.537559 ))
    * Do not need all of the metagenomes and viromes, there are a lot, will just use the three largest assemblies from each
    * All of these samples were assembled with MEGAHIT with the longest kmer set to 127.
    * Soil microbial and viral communities from microcosms at the University of Lyon, Ecully, France (JGI) from [JGI](https://doi.org/10.46936/10.25585/60008876) (Christina Hazard PI)

* Complete viral genomes: 
    * Soil: random subsamples of 1000 high-quality/complete genomes from [IMG/VR v4](https://img.jgi.doe.gov/cgi-bin/vr/main.cgi) with soil ecosystem classifications 
    * Marine: '' with marine ecosystem classifications 
    * Host-associated: '' with human digestive system ecosystem classifications

## Organize and obtain datasets

### Metagenomes and viromes
Downloaded from the [JGI Data Portal](https://data.jgi.doe.gov/).

In [2]:
from pathlib import Path
ROOT_DIR = Path("/storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks")
GENOME_DIR = ROOT_DIR.joinpath("benchmark_genomes")
METAG_DIR = GENOME_DIR.joinpath("metagenomes")
VIROME_DIR = GENOME_DIR.joinpath("viromes")

In [5]:
! ls -lh {METAG_DIR} {VIROME_DIR}
! grep -c ">" {METAG_DIR}/*.fna {VIROME_DIR}/*.fna

/storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/benchmark_genomes/metagenomes:
total 2.5G
lrwxrwxrwx 1 kosmopoulos kosmopoulos  105 Jan  7 12:17 freshwater_Ga0485157_contigs.fna -> /storage2/scratch/xhuang495/Pro1_compar_meta_virom/contigs/00.total_seqs/freshwater/Ga0485157_contigs.fna
lrwxrwxrwx 1 kosmopoulos kosmopoulos  105 Jan  7 12:18 freshwater_Ga0485158_contigs.fna -> /storage2/scratch/xhuang495/Pro1_compar_meta_virom/contigs/00.total_seqs/freshwater/Ga0485158_contigs.fna
lrwxrwxrwx 1 kosmopoulos kosmopoulos  105 Jan  7 12:18 freshwater_Ga0485159_contigs.fna -> /storage2/scratch/xhuang495/Pro1_compar_meta_virom/contigs/00.total_seqs/freshwater/Ga0485159_contigs.fna
lrwxrwxrwx 1 kosmopoulos kosmopoulos  105 Sep 24  2025 human_gut_SRR9162900_contigs.fna -> /storage2/scratch/xhuang495/Pro1_compar_meta_virom/contigs/00.total_seqs/human_gut/SRR9162900_contigs.fna
lrwxrwxrwx 1 kosmopoulos kosmopoulos  105 Sep 24  2025 human_gut_SRR9162906_contigs.fna -> /storage2/scratch/xh

### Viral genomes

Sample complete viral genomes from the [IMG/VR v4 database](https://genome.jgi.doe.gov/portal/IMG_VR/IMG_VR.home.html)

In [3]:
import os
os.environ["POLARS_MAX_THREADS"] = str(50)
import polars as pl
pl.Config.set_fmt_str_lengths(100)

polars.config.Config

In [4]:
IMGVR_DIR = Path("/storage2/databases/IMGVR/IMGVR_V4/")

In [8]:
imgvr_info = (
    pl.read_csv(
        IMGVR_DIR.joinpath("IMGVR_all_Sequence_information.tsv"),
        separator="\t",
        schema_overrides={
            "Scaffold_oid": pl.Utf8, 
            "Taxon_oid":    pl.Utf8,
        },
        null_values=["NA", ";;;;;;", ";;;;;", ";;;;", ";;;"]
    )
    .filter(
        (pl.col("MIUViG quality") == "High-quality") &
        (pl.col("Topology") != "GVMAG") &
        (pl.col("Estimated contamination") == 0.0) &
        (pl.col("Host taxonomy prediction").is_not_null())
    )
    .with_columns(
        pl.when(
            pl.col("Topology") == "Provirus"
        )
        .then(
            pl.concat_str(
                ["UVIG", "Taxon_oid", "Scaffold_oid", "Coordinates ('whole' if the UViG is the entire contig)"],
                separator="|"
                )
        )
        .otherwise(
            pl.concat_str(["UVIG", "Taxon_oid", "Scaffold_oid"], separator="|")
        )
        .alias("sequence")
    )
    .select(
        pl.col("sequence"),
        pl.col("UVIG"),
        pl.col("Taxon_oid"),
        pl.col("Scaffold_oid"),
        pl.col("Ecosystem classification"),
        pl.col("Length"),
        pl.col("Topology"),
        pl.col("Host taxonomy prediction"),
    )
    .sort("sequence")
)

In [9]:
imgvr_info

sequence,UVIG,Taxon_oid,Scaffold_oid,Ecosystem classification,Length,Topology,Host taxonomy prediction
str,str,str,str,str,i64,str,str
"""IMGVR_UViG_2020627000_000021|2020627000|VrWwEF_contig52410""","""IMGVR_UViG_2020627000_000021""","""2020627000""","""VrWwEF_contig52410""","""Engineered;Wastewater;Nutrient removal;Dissolved organics (anaerobic)""",34219,"""Linear""","""d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Bacteroidales;f__Rikenellaceae;g__Tidjanibacter;"""
"""IMGVR_UViG_2022920007_000001|2022920007|YNPsite14_CeleraDRAF_scf1118686615532""","""IMGVR_UViG_2022920007_000001""","""2022920007""","""YNPsite14_CeleraDRAF_scf1118686615532""","""Environmental;Aquatic;Thermal springs;Hot (42-90C)""",34310,"""Linear""","""d__Archaea;p__Thermoproteota;c__Thermoproteia;;;;"""
"""IMGVR_UViG_2029527003_000023|2029527003|APTF_contig46037|167350-219702""","""IMGVR_UViG_2029527003_000023""","""2029527003""","""APTF_contig46037""","""Environmental;Terrestrial;Nest;Insects nest""",52353,"""Provirus""","""d__Bacteria;p__Proteobacteria;c__Gammaproteobacteria;o__Enterobacterales;f__Enterobacteriaceae;g__Se…"
"""IMGVR_UViG_2029527005_000024|2029527005|ACOFG987_contig42945|4748-48897""","""IMGVR_UViG_2029527005_000024""","""2029527005""","""ACOFG987_contig42945""","""Environmental;Terrestrial;Nest;Insects nest""",44150,"""Provirus""","""d__Bacteria;p__Proteobacteria;c__Gammaproteobacteria;o__Enterobacterales;f__Enterobacteriaceae;;"""
"""IMGVR_UViG_2029527005_000025|2029527005|ACOFG987_contig46306|2-31265""","""IMGVR_UViG_2029527005_000025""","""2029527005""","""ACOFG987_contig46306""","""Environmental;Terrestrial;Nest;Insects nest""",31264,"""Provirus""","""d__Bacteria;p__Proteobacteria;c__Gammaproteobacteria;o__Enterobacterales;f__Enterobacteriaceae;g__Sa…"
…,…,…,…,…,…,…,…
"""IMGVR_UViG_7000000740_000333|7000000740|SRS064276_LANL_scaffold_40179|28691-63532""","""IMGVR_UViG_7000000740_000333""","""7000000740""","""SRS064276_LANL_scaffold_40179""","""Host-associated;Human;Digestive system;Large intestine""",34842,"""Provirus""","""d__Bacteria;p__Firmicutes_A;c__Clostridia;o__Oscillospirales;f__Ruminococcaceae;g__Ruminococcus_D;"""
"""IMGVR_UViG_7000000742_000014|7000000742|SRS015044_WUGC_scaffold_34979""","""IMGVR_UViG_7000000742_000014""","""7000000742""","""SRS015044_WUGC_scaffold_34979""","""Host-associated;Human;Digestive system;Oral cavity""",31801,"""Linear""","""d__Bacteria;p__Firmicutes;c__Bacilli;o__Lactobacillales;f__Aerococcaceae;g__Abiotrophia;"""
"""IMGVR_UViG_7000000742_000026|7000000742|SRS015044_WUGC_scaffold_54346""","""IMGVR_UViG_7000000742_000026""","""7000000742""","""SRS015044_WUGC_scaffold_54346""","""Host-associated;Human;Digestive system;Oral cavity""",40710,"""Linear""","""d__Bacteria;p__Proteobacteria;c__Gammaproteobacteria;o__Burkholderiales;f__Neisseriaceae;g__Neisseri…"


In [5]:
N_VIRUS = 1000
SEED = 20260107

In [11]:
imgvr_soil = (
    imgvr_info
    .filter(pl.col("Ecosystem classification").str.contains("Soil", literal=True))
    .get_column("sequence")
    .unique()
    .sort() # Sort before sampling to ensure reproducibility
    .sample(n=N_VIRUS, seed=SEED)
    .to_list()
)

imgvr_soil[:5], len(imgvr_soil)

(['IMGVR_UViG_2786546141_000001|2786546141|2786548633',
  'IMGVR_UViG_2927261028_000002|2927261028|2927261028|1278208-1327267',
  'IMGVR_UViG_641228486_000003|641228486|641228511|3612634-3657896',
  'IMGVR_UViG_2911215477_000001|2911215477|2911215477|64774-167258',
  'IMGVR_UViG_3300028552_000019|3300028552|Ga0302149_1000056'],
 1000)

In [12]:
imgvr_marine = (
    imgvr_info
    .filter(pl.col("Ecosystem classification").str.contains("Marine", literal=True))
    .get_column("sequence")
    .unique()
    .sort() # Sort before sampling to ensure reproducibility
    .sample(n=N_VIRUS, seed=SEED)
    .to_list()
)

imgvr_marine[:5], len(imgvr_marine)

(['IMGVR_UViG_2835163134_000006|2835163134|2835163186|133193-171013',
  'IMGVR_UViG_3300017593_000534|3300017593|Ga0187538_132574',
  'IMGVR_UViG_641522618_000003|641522618|641522729|942494-975653',
  'IMGVR_UViG_3300048587_001116|3300048587|Ga0498925_0002331|8768-49391',
  'IMGVR_UViG_2675903148_000001|2675903148|2675940066'],
 1000)

In [13]:
imgvr_gut = (
    imgvr_info
    .filter(pl.col("Ecosystem classification").str.contains("Human;Digestive system;Large intestine", literal=True))
    .get_column("sequence")
    .unique()
    .sort() # Sort before sampling to ensure reproducibility
    .sample(n=N_VIRUS, seed=SEED)
    .to_list()
)

imgvr_gut[:5], len(imgvr_gut)

(['IMGVR_UViG_3300029842_000384|3300029842|Ga0245273_100021|251277-310967',
  'IMGVR_UViG_3300045988_052645|3300045988|Ga0495776_000177',
  'IMGVR_UViG_3300045988_183987|3300045988|Ga0495776_066373',
  'IMGVR_UViG_3300045988_065384|3300045988|Ga0495776_020146',
  'IMGVR_UViG_3300045988_042160|3300045988|Ga0495776_028777'],
 1000)

In [14]:
genome_to_env = {}
for g in imgvr_soil:
    genome_to_env[g] = "soil"
for g in imgvr_marine:
    genome_to_env[g] = "marine"
for g in imgvr_gut:
    genome_to_env[g] = "gut"

In [15]:
len(genome_to_env)

3000

In [6]:
IMGVR_FASTA = IMGVR_DIR.joinpath("high_quality/single-contig.fna")
VIRUS_GENOME_DIR = GENOME_DIR.joinpath("complete_virus_genomes")
SOIL_FASTA = VIRUS_GENOME_DIR.joinpath("virus_genomes_soil.fna")
MARINE_FASTA = VIRUS_GENOME_DIR.joinpath("virus_genomes_marine.fna")
GUT_FASTA = VIRUS_GENOME_DIR.joinpath("virus_genomes_gut.fna")

In [17]:
from pyfastatools import Parser, write_fasta

for p in [SOIL_FASTA, MARINE_FASTA, GUT_FASTA]:
    Path(p).parent.mkdir(parents=True, exist_ok=True)

def norm_id(s):
    return s.split()[0]

expected = set(genome_to_env.keys())
found = set()
written = 0

with open(SOIL_FASTA, "w") as f_soil, \
     open(MARINE_FASTA, "w") as f_marine, \
     open(GUT_FASTA, "w") as f_gut:

    for rec in Parser(IMGVR_FASTA):
        rid = norm_id(rec.header.name)
        if rid not in expected:
            continue
        env = genome_to_env[rid]
        if env == "soil":
            write_fasta(rec, f_soil)
        elif env == "marine":
            write_fasta(rec, f_marine)
        elif env == "gut":
            write_fasta(rec, f_gut)
        else:
            continue
        found.add(rid)
        written += 1

missing = sorted(expected - found)
print(f"Found {len(found)} of {len(expected)} expected genomes, wrote {written} records")
print(f"Missing count: {len(missing)}")

Found 3000 of 3000 expected genomes, wrote 3000 records
Missing count: 0


In [18]:
! head -n 5 {SOIL_FASTA} {MARINE_FASTA} {GUT_FASTA}
! grep -c ">" {SOIL_FASTA} {MARINE_FASTA} {GUT_FASTA}

==> /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/benchmark_genomes/complete_virus_genomes/virus_genomes_soil.fna <==
>IMGVR_UViG_2507262019_000001|2507262019|2507271754|10680-27242 
ATGAAAATATTAAAAATTGAGAACGGCCAAGGTTACTTTGCAACCGCCGAGGGTGAGTATGAAACTATCGACAAG
ATTGACAAGGCTATCCTGTCGAGGTTAGTTAACTCTGCTTTGGAAGATGGATTTAAGATAGACGAGTACAGCGAG
ATAGAACTACAAAACCAAGCGCATCAAATAATATATAAGAGCATCAGTGAAAAACTCTTGGATCTCCACCAAAAA
CGTAGACAGTTTCGCGACGAATCAGACCGACTATATCTAGATGCTTATGAGAAGTACAAATCCTAAGACGTCTAT

==> /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/benchmark_genomes/complete_virus_genomes/virus_genomes_marine.fna <==
>IMGVR_UViG_2551306319_000001|2551306319|2551378208 
AAATCCGGCGGCGTGTCGAGCGCCTTTAAATTGGTGACCGTCAGTCTCGGCGGCCAGATGCGACCACATTCATCG
ATAAAGAGAAACGCGTCCTTTCTCGCCCAGTGCCAAAAACGCGCCATCGTTAAGCGACCGTCAGGATGGTCTGTA
TCAATAAACTCGATACTGATGTCCGAGACATCCATTTTTAAGTACTTAGCCATGCGTTCAAGGTTTAAGCCTCGC
ACATTCGTGATGATGTGACGGCCTGACTTAATCGCCGGCAGCAGACGAAGCCATAATGCCCCTGACGTTTTATAA

==> /sto

## Independent prediction of viral regions

Metagenomes and viromes will contain non-viral sequences, so they need to be sorted out. And the virus genomes can still contain some contamination. To provide an independent assessment of viral/nonviral "ground-truth", geNomad v1.11.0 will be used. Even though geNomad isn't perfect, running it with [score calibration](https://portal.nersc.gov/genomad/score_calibration.html) should help filter the results to a desirable FDR (< 0.1).

### Execute genomad for each input using Snakemake v7.32.4

All necessary snakemake files are located in `accessory_scripts`.

In [7]:
SCRIPTS_DIR = Path("./accessory_scripts")

In [8]:
! cat {SCRIPTS_DIR}/genomad.config.yaml

input_base: /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/benchmark_genomes
output_base: /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/benchmark_genomes/genomad_outputs
genomad_db: /storage2/scratch/kosmopoulos/databases/genomad/genomad_db/
threads: 32


In [9]:
! cat {SCRIPTS_DIR}/genomad.snakefile

import os
from pathlib import Path

configfile: "./accessory_scripts/genomad.config.yaml"

INPUT_BASE = Path(config["input_base"]).resolve()
OUTPUT_BASE = Path(config["output_base"]).resolve()

# Pattern: <input_base>/<subset>/<sample>.fna
# subset ∈ {"viromes","metagenomes","complete_virus_genomes"}
INP_GLOB = str(INPUT_BASE / "*" / "*.fna")

def outdir(subset, sample):
    return OUTPUT_BASE / subset / sample

# Discover all .fna inputs up front
INPUT_FILES = sorted(Path(p) for p in map(str, __import__("glob").glob(INP_GLOB)))
INPUT_FILES = [p for p in INPUT_FILES if p.parent.name in {"viromes","metagenomes","complete_virus_genomes"}]
# Build expected output directories (as Snakemake "directory" targets) from discovered inputs
OUT_DIRS = [outdir(p.parent.name, p.stem) for p in INPUT_FILES]

rule all:
    input:
        [directory(p) for p in OUT_DIRS]

rule genomad:
    input:
        contigs=os.path.join(config["input_base"], "{subset}", "{sample}.fna")
    output:
        directory

**Note** that the `--relaxed` option is being used so we can accept and get data on all contigs and do our own filtering, later.

    nohup snakemake \
        --use-conda \
        -j 64 \
        --snakefile ./accessory_scripts/genomad.snakefile

## Run CheckAMG, DRAM-V, and VIBRANT

Some things to consider:

* Each tool has confidence cutoffs:
    * CheckAMG has 3 confidence levels (high, medium, low)
    * DRAM-V has multiple 'flags' for AMGs, but the [documentation](https://github.com/WrightonLabCSU/DRAM/wiki/1.-How-DRAM-Works#dram-v-in-detail) says that putative AMGs have flags "cat 1-3, M, F" and non-AMGs have flags "cat 4, V, A, P, B", the "T" flag (indicating a nearby transposon) isn't recommended in the documentation one way or the other, but users sometimes discard AMGs with this flag
    * VIBRANT does not assign AMG-level cutoffs, but labels virus predictions as "complete circular," "high," "medium," and "low quality draft"

* Class imbalances:
    * There will be imbalances in the number of viral/non-viral proteins and the number of AMGs/non-AMGs in each dataset

* Protein-protein comparisons
    * Each tool, when run with default settings, will produce different proteomes, since they each do their own gene calling, and do this with different methods. It would be ideal to use the exact same proteins on each tool.
    * Fortunately, both CheckAMG and VIBRANT can start from protein inputs. DRAM-V cannot, but **DRAM-V can be run first, then CheckAMG and VIBRANT can run using the protein outputs from DRAM-V**.

All necessary files and scripts used to run these tools are located in `accessory_scripts`.

In [10]:
TOOL_OUTDIR = ROOT_DIR.joinpath("metabolism_benchmark")

### Generate the `VIRSorter_affi-contigs.tab` files for DRAM-V
Need to generate these with VirSorter2 (used v2.2)

In [11]:
! cat {SCRIPTS_DIR}/virsorter.config.yaml

input_base: /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/benchmark_genomes
output_base: /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/virsorter_outputs
virsorter_db: /storage2/scratch/kosmopoulos/databases/VirSorter2_db
threads: 16


In [12]:
! cat {SCRIPTS_DIR}/virsorter.snakefile

import os
from pathlib import Path

configfile: "./accessory_scripts/virsorter.config.yaml"

INPUT_BASE = Path(config["input_base"]).resolve()
OUTPUT_BASE = Path(config["output_base"]).resolve()

# Pattern: <input_base>/<subset>/<sample>.fna
# subset ∈ {"viromes","metagenomes","complete_virus_genomes"}
INP_GLOB = str(INPUT_BASE / "*" / "*.fna")

def outdir(subset, sample):
    return OUTPUT_BASE / subset / sample

# Discover all .fna inputs up front
INPUT_FILES = sorted(Path(p) for p in map(str, __import__("glob").glob(INP_GLOB)))
INPUT_FILES = [p for p in INPUT_FILES if p.parent.name in {"viromes","metagenomes","complete_virus_genomes"}]
# Build expected output directories (as Snakemake "directory" targets) from discovered inputs
OUT_DIRS = [outdir(p.parent.name, p.stem) for p in INPUT_FILES]

rule all:
    input:
        [directory(p) for p in OUT_DIRS]

rule virsorter:
    input:
        contigs=os.path.join(config["input_base"], "{subset}", "{sample}.fna")
    output:
        direc

    nohup snakemake \
        --use-conda \
        -j 160 \
        --snakefile ./accessory_scripts/virsorter.snakefile

### Run DRAM-v
Using DRAM-v v1.5.0 with DBs:

In [1]:
! conda run -n DRAM DRAM-setup.py print_config

Processed search databases
KEGG db: None
KOfam db: /storage2/databases/dram-latest/v1.5.0/kofam_profiles.hmm
KOfam KO list: /storage2/databases/dram-latest/v1.5.0/kofam_ko_list.tsv
UniRef db: /storage2/databases/dram-latest/v1.5.0/uniref90.20250127.mmsdb
Pfam db: /storage2/databases/dram-latest/v1.5.0/pfam.mmspro
dbCAN db: /storage2/databases/dram-latest/v1.5.0/dbCAN-HMMdb-V11.txt
RefSeq Viral db: /storage2/databases/dram-latest/v1.5.0/refseq_viral.20250128.mmsdb
MEROPS peptidase db: /storage2/databases/dram-latest/v1.5.0/peptidases.20250128.mmsdb
VOGDB db: /storage2/databases/dram-latest/v1.5.0/vog_latest_hmms.txt
CAMPER HMM db: None
CAMPER FASTA db: None
CAMPER HMM cutoffs: None
CAMPER FASTA cutoffs: None

Descriptions of search database entries
Pfam hmm dat: /storage2/databases/dram-latest/v1.5.0/Pfam-A.hmm.dat.gz
dbCAN family activities: /storage2/databases/dram-latest/v1.5.0/CAZyDB.08062022.fam-activities.txt
VOG annotations: /storage2/databases/dram-latest/v1.5.0/vog_annotations_

Using default settings **except** add `--max_auxiliary_score 4` to ensure all possible AMG predictions are included in the outputs. These will be filtered later.

In [13]:
! cat {SCRIPTS_DIR}/dramv.config.yaml

input_base: /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/virsorter_outputs
output_base: /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/dramv_outputs
threads: 4


In [14]:
! cat {SCRIPTS_DIR}/dramv.snakefile

import os
from pathlib import Path

configfile: "./accessory_scripts/dramv.config.yaml"

INPUT_BASE = Path(config["input_base"]).resolve()
OUTPUT_BASE = Path(config["output_base"]).resolve()

def discover_samples():
    for subset_dir in INPUT_BASE.iterdir():
        if not subset_dir.is_dir():
            continue
        for sample_dir in subset_dir.iterdir():
            if not sample_dir.is_dir():
                continue
            fd = sample_dir / "for-dramv"
            if fd.is_dir():
                fas = list(fd.glob("*.fa"))
                tabs = list(fd.glob("*.tab"))
                if len(fas) == 1 and len(tabs) == 1:
                    yield (subset_dir.name, sample_dir.name)

SAMPLES = sorted(set(discover_samples()))
ALL_DISTILLED = [
    str(OUTPUT_BASE / subset / sample / "distilled")
    for subset, sample in SAMPLES
]

# remove complete host genomes
for i, path in enumerate(ALL_DISTILLED):
    if "host_genomes" in path:
        ALL_DISTILLED.pop(i)

rule all:
  

    nohup snakemake \
        --use-conda \
        -j 48 \
        --snakefile ./accessory_scripts/dramv.snakefile

### Gather and reformat DRAM-V proteins for inputs to VIBRANT and CheckAMG

Need to take the prodigal outputs from DRAM-V and reformat them. VirSorter2 reformats sequence headers, which went into DRAM-V, so this needs to be resolved to match their original headers and to be in acceptable input formats for VIBRANT and CheckAMG.

In [15]:
DRAMV_OUTDIR = TOOL_OUTDIR.joinpath("dramv_outputs")

In [45]:
from pathlib import Path

def discover_samples():
    for subset_dir in DRAMV_OUTDIR.iterdir():
        if not subset_dir.is_dir() or subset_dir.name not in [
            "complete_host_genomes", "complete_virus_genomes", "metagenomes", "viromes"
        ]:
            continue
        for sample_dir in subset_dir.iterdir():
            if not sample_dir.is_dir():
                continue
            genes = list(sample_dir.glob("genes.*"))
            if genes:
                yield (subset_dir.name, sample_dir.name)

SAMPLES = sorted(set(discover_samples()))
SAMPLES

[('complete_virus_genomes', 'virus_genomes_gut'),
 ('complete_virus_genomes', 'virus_genomes_marine'),
 ('complete_virus_genomes', 'virus_genomes_soil'),
 ('metagenomes', 'freshwater_Ga0485157_contigs'),
 ('metagenomes', 'freshwater_Ga0485158_contigs'),
 ('metagenomes', 'freshwater_Ga0485159_contigs'),
 ('metagenomes', 'human_gut_SRR9162900_contigs'),
 ('metagenomes', 'human_gut_SRR9162906_contigs'),
 ('metagenomes', 'human_gut_SRR9162908_contigs'),
 ('metagenomes', 'soil_T42_15_1_55'),
 ('metagenomes', 'soil_T42_15_2_56'),
 ('metagenomes', 'soil_T42_15_3_57'),
 ('viromes', 'freshwater_Ga0485173_contigs'),
 ('viromes', 'freshwater_Ga0485174_contigs'),
 ('viromes', 'freshwater_Ga0485175_contigs'),
 ('viromes', 'human_gut_SRR9161502_contigs'),
 ('viromes', 'human_gut_SRR9161506_contigs'),
 ('viromes', 'human_gut_SRR9161509_contigs'),
 ('viromes', 'soil_T42_15_1_43'),
 ('viromes', 'soil_T42_15_2_44'),
 ('viromes', 'soil_T42_15_3_45')]

In [46]:
def genes_faa(subset, sample):
    p = DRAMV_OUTDIR / subset / sample
    faas = sorted(p.glob("genes.faa"))
    assert len(faas) == 1, f"Expected 1 .faa in {p}, found {len(faas)}"
    return str(faas[0])

def genes_gff(subset, sample):
    p = DRAMV_OUTDIR / subset / sample 
    gffs = sorted(p.glob("genes.gff"))
    assert len(gffs) == 1, f"Expected 1 .gff in {p}, found {len(gffs)}"
    return str(gffs[0])

In [ ]:
from collections import defaultdict
from pyfastatools import Parser

dramv_genes = defaultdict(dict)

for subset, sample in SAMPLES:
    faa_path = genes_faa(subset, sample)
    gff_path = genes_gff(subset, sample)
    
    dramv_genes[subset][sample] = {
            "faa": faa_path,
            "gff": gff_path,
            "records": {}
    }
    
    for record in Parser(faa_path):
        dramv_genes[subset][sample]["records"][record.header.name] = record

In [ ]:
import re

dramv_genes_reformatted = dramv_genes.copy()

def parse_gff_to_id_map(gff_lines):
    id_map = {}
    for line in gff_lines:
        fields = line.strip().split('\t')
        if len(fields) < 9:
            continue
        attr = fields[8]
        m = re.search(r'ID=([^;]+)', attr)
        if m:
            id_map[m.group(1)] = fields
    return id_map

# records for mapping dataframe
mapping_records = []

for subset, samples in dramv_genes_reformatted.items():
    for sample, objs in samples.items():
        # Load GFF
        if isinstance(objs["gff"], str):
            with open(objs["gff"], "r") as gff_fh:
                gff_lines = [line.strip() for line in gff_fh if line.strip() and not line.startswith("#")]
        else:
            gff_lines = objs["gff"]
        id2fields = parse_gff_to_id_map(gff_lines)

        # store per-scaffold gene count and max position
        scaf_counts = defaultdict(int)
        scaf_maxend = defaultdict(int)

        # First pass to collect scaffold stats
        for rid, record in objs["records"].items():
            gff_fields = id2fields.get(record.header.name)
            if gff_fields is None:
                continue
            start, end = int(gff_fields[3]), int(gff_fields[4])
            scaffold = record.header.name.rsplit("_", 1)[0]
            scaf_counts[scaffold] += 1
            scaf_maxend[scaffold] = max(scaf_maxend[scaffold], start, end)

        # Second pass to rename and record mappings
        for rid, record in list(objs["records"].items()):
            old_gene = record.header.name
            dramv_scaffold, dramv_addn = old_gene.rsplit("_", 1)[0], old_gene.rsplit("-", 1)[-1]
            new_scaffold, gene_num = dramv_scaffold.rsplit("-", 1)[0], dramv_addn.rsplit("_", 1)[-1]

            gff_fields = id2fields.get(old_gene)
            if gff_fields is None:
                print(f"  No GFF match found for {old_gene}")
                continue

            start = int(gff_fields[3])
            end = int(gff_fields[4])
            strand = gff_fields[6]
            frame = 1 if strand == "+" else -1
            metadata = gff_fields[8]

            new_gene = f"{new_scaffold}_{gene_num}"
            new_id = f"{new_gene} # {start} # {end} # {frame} # {metadata}"

            # update FASTA header
            record.header.name = new_gene
            record.header.desc = f"# {start} # {end} # {frame} # {metadata}"
            dramv_genes_reformatted[subset][sample]["records"][rid] = record

            # scaffold stats
            scaf_gene_count = scaf_counts.get(dramv_scaffold, 0)
            scaf_len = scaf_maxend.get(dramv_scaffold, 0)

            mapping_records.append((
                subset, sample,
                old_gene, new_gene,
                dramv_scaffold, new_scaffold,
                int(gene_num),
                start, end, frame,
                scaf_gene_count, scaf_len,
                metadata
            ))


In [ ]:
gene_id_mapping = pl.DataFrame(
    mapping_records,
    schema=[
        "subset", "sample",
        "old_gene", "new_gene",
        "old_scaffold", "new_scaffold",
        "gene_number", "start", "end", "frame",
        "scaffold_n_genes", "scaffold_len_bases",
        "metadata"
    ]
)

gene_id_mapping

/storage2/scratch/kosmopoulos/miniconda3/envs/peatlands_env/lib/python3.10/functools.py:889: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  return dispatch(args[0].__class__)(*args, **kw)


subset,sample,old_gene,new_gene,old_scaffold,new_scaffold,gene_number,start,end,frame,scaffold_n_genes,scaffold_len_bases,metadata
str,str,str,str,str,str,i64,i64,i64,i64,i64,i64,str
"""complete_host_genomes""","""host_genomes_gut""","""391036.SAMN02367296.CP007474-cat_2_1""","""391036.SAMN02367296.CP007474_1""","""391036.SAMN02367296.CP007474-cat_2""","""391036.SAMN02367296.CP007474""",1,1,1005,1,910,1148482,"""ID=391036.SAMN02367296.CP007474-cat_2_1;conf=100.00;cscore=98.58;Dbxref=""ko:K01599"";gc_cont=0.338;pa…"
"""complete_host_genomes""","""host_genomes_gut""","""391036.SAMN02367296.CP007474-cat_2_2""","""391036.SAMN02367296.CP007474_2""","""391036.SAMN02367296.CP007474-cat_2""","""391036.SAMN02367296.CP007474""",2,1102,1926,1,910,1148482,"""ID=391036.SAMN02367296.CP007474-cat_2_2;conf=100.00;cscore=48.58;Dbxref=""ko:K02276"";gc_cont=0.348;pa…"
"""complete_host_genomes""","""host_genomes_gut""","""391036.SAMN02367296.CP007474-cat_2_3""","""391036.SAMN02367296.CP007474_3""","""391036.SAMN02367296.CP007474-cat_2""","""391036.SAMN02367296.CP007474""",3,1928,2398,1,910,1148482,"""ID=391036.SAMN02367296.CP007474-cat_2_3;conf=100.00;cscore=45.39;Dbxref=""vogdb:VOG40486"";gc_cont=0.3…"
"""complete_host_genomes""","""host_genomes_gut""","""391036.SAMN02367296.CP007474-cat_2_4""","""391036.SAMN02367296.CP007474_4""","""391036.SAMN02367296.CP007474-cat_2""","""391036.SAMN02367296.CP007474""",4,2395,2985,-1,910,1148482,"""ID=391036.SAMN02367296.CP007474-cat_2_4;conf=99.92;cscore=30.30;Dbxref=""ko:K08591"";gc_cont=0.286;par…"
"""complete_host_genomes""","""host_genomes_gut""","""391036.SAMN02367296.CP007474-cat_2_5""","""391036.SAMN02367296.CP007474_5""","""391036.SAMN02367296.CP007474-cat_2""","""391036.SAMN02367296.CP007474""",5,3081,3920,1,910,1148482,"""ID=391036.SAMN02367296.CP007474-cat_2_5;conf=100.00;cscore=112.25;Dbxref=""ko:K00767"";gc_cont=0.298;p…"
…,…,…,…,…,…,…,…,…,…,…,…,…
"""viromes""","""soil_T42_15_3_45""","""scaffold_7091_c1__lt2gene-cat_1_1""","""scaffold_7091_c1__lt2gene_1""","""scaffold_7091_c1__lt2gene-cat_1""","""scaffold_7091_c1__lt2gene""",1,1,2931,-1,1,2931,"""ID=scaffold_7091_c1__lt2gene-cat_1_1;conf=99.99;cscore=347.23;Dbxref=""viral:YP_010091592.1"";gc_cont=…"
"""viromes""","""soil_T42_15_3_45""","""scaffold_7858_c1__lt2gene-cat_1_1""","""scaffold_7858_c1__lt2gene_1""","""scaffold_7858_c1__lt2gene-cat_1""","""scaffold_7858_c1__lt2gene""",1,1,1146,-1,2,2798,"""ID=scaffold_7858_c1__lt2gene-cat_1_1;conf=100.00;cscore=160.20;gc_cont=0.695;partial=10;rbs_motif=GG…"
"""viromes""","""soil_T42_15_3_45""","""scaffold_7858_c1__lt2gene-cat_1_2""","""scaffold_7858_c1__lt2gene_2""","""scaffold_7858_c1__lt2gene-cat_1""","""scaffold_7858_c1__lt2gene""",2,1716,2798,1,2,2798,"""ID=scaffold_7858_c1__lt2gene-cat_1_2;conf=100.00;cscore=173.16;gc_cont=0.635;partial=01;rbs_motif=No…"


In [16]:
DRAMV_GENES_REFORMAT_DIR = TOOL_OUTDIR.joinpath("dramv_genes_reformatted")
DRAMV_GENES_REFORMAT_DIR.mkdir(exist_ok=True)
GENE_ID_MAPPING = DRAMV_GENES_REFORMAT_DIR.joinpath("gene_id_mapping.parquet")

In [ ]:
gene_id_mapping.write_parquet(GENE_ID_MAPPING)

In [ ]:
import os

for subset, samples in dramv_genes_reformatted.items():
    for sample, objs in samples.items():
        os.makedirs(DRAMV_GENES_REFORMAT_DIR / subset / sample, exist_ok=True)
        out_faa = DRAMV_GENES_REFORMAT_DIR / subset / sample / "genes_reformatted.faa"
        print(f"Writing {len(objs['records'].items())} reformatted genes to {out_faa}")
        n = 0
        with open(out_faa, "w") as out_fh:
            for rid, record in objs["records"].items():
                out_fh.write(f">{record.header.name} {record.header.desc}\n")
                seq = record.seq
                for i in range(0, len(seq), 80):
                    out_fh.write(seq[i:i+80] + "\n")
                n += 1
        print(f"  Wrote {n} records")

Writing 1361999 reformatted genes to /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/benchmark_genomes/dramv_genes_reformatted/complete_host_genomes/host_genomes_gut/genes_reformatted.faa
  Wrote 1361999 records
Writing 1614495 reformatted genes to /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/benchmark_genomes/dramv_genes_reformatted/complete_host_genomes/host_genomes_marine/genes_reformatted.faa
  Wrote 1614495 records
Writing 2289113 reformatted genes to /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/benchmark_genomes/dramv_genes_reformatted/complete_host_genomes/host_genomes_soil/genes_reformatted.faa
  Wrote 2289113 records
Writing 74398 reformatted genes to /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/benchmark_genomes/dramv_genes_reformatted/complete_virus_genomes/virus_genomes_gut/genes_reformatted.faa
  Wrote 74398 records
Writing 55262 reformatted genes to /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/benchmark_ge

### Run VIBRANT
Using VIBRANT v1.2.1 with databases:

* KEGG KOfam v91 (2019-08-10) (accessed 2024-02-01)
* Pfam-A v32.0 (2018-10) (accessed 2024-02-01)
* VOGDB v94 (2023-08-22) (accessed 2024-02-01)

In [17]:
! cat {SCRIPTS_DIR.joinpath("vibrant.config.yaml")}

input_base: /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/dramv_genes_reformatted
output_base: /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/vibrant_outputs
vibrant_db: /storage2/databases/VIBRANT/databases/
vibrant_files: /storage2/scratch/kosmopoulos/software/VIBRANT/files/
threads: 16


In [18]:
! cat {SCRIPTS_DIR.joinpath("vibrant.snakefile")}

import os
from pathlib import Path

configfile: "./accessory_scripts/vibrant.config.yaml"

INPUT_BASE = Path(config["input_base"]).resolve()
OUTPUT_BASE = Path(config["output_base"]).resolve()

# Pattern: <input_base>/<subset>/<sample>.fna
# subset ∈ {"viromes","metagenomes","complete_virus_genomes"}
INP_GLOB = str(INPUT_BASE / "*" / "*" / "genes_reformatted.faa")

def outdir(subset, sample):
    return OUTPUT_BASE / subset / sample

# Discover all .faa inputs up front
INPUT_FILES = sorted(Path(p) for p in map(str, __import__("glob").glob(INP_GLOB)))
INPUT_FILES = [p for p in INPUT_FILES if p.parent.parent.name in {"viromes","metagenomes","complete_virus_genomes"}]
# Build expected output directories (as Snakemake "directory" targets) from discovered inputs
OUT_DIRS = [outdir(p.parent.parent.name, p.parent.name) for p in INPUT_FILES]

rule all:
    input:
        [directory(p) for p in OUT_DIRS]

rule vibrant:
    input:
        ptns=os.path.join(config["input_base"], "{subset}", "{sam

    nohup snakemake \
        --use-conda \
        -j 48 \
        --snakefile ./accessory_scripts/vibrant.snakefile

### Run CheckAMG annotate
Using CheckAMG v1.1 with CheckAMG db v1.1

In [19]:
! cat {SCRIPTS_DIR.joinpath("checkamg_annotate.config.yaml")}

input_base: /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/dramv_genes_reformatted
output_base: /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/checkamg_annotate_v1.1_outputs
checkamg_db: /storage2/scratch/kosmopoulos/software/CheckAMG/notebooks/CheckAMG_annotate_db_v1.1_20260316
threads: 32


In [20]:
! cat {SCRIPTS_DIR.joinpath("checkamg_annotate.snakefile")}

import os
from pathlib import Path

configfile: "./accessory_scripts/checkamg_annotate.config.yaml"

INPUT_BASE = Path(config["input_base"]).resolve()
OUTPUT_BASE = Path(config["output_base"]).resolve()

# Pattern: <input_base>/<subset>/<sample>.fna
# subset ∈ {"viromes","metagenomes","complete_virus_genomes"}
INP_GLOB = str(INPUT_BASE / "*" / "*" / "genes_reformatted.faa")

def outdir(subset, sample):
    return OUTPUT_BASE / subset / sample

# Discover all .faa inputs up front
INPUT_FILES = sorted(Path(p) for p in map(str, __import__("glob").glob(INP_GLOB)))
INPUT_FILES = [p for p in INPUT_FILES if p.parent.parent.name in {"viromes","metagenomes","complete_virus_genomes"}]
# Build expected output directories (as Snakemake "directory" targets) from discovered inputs
OUT_DIRS = [outdir(p.parent.parent.name, p.parent.name) for p in INPUT_FILES]

rule all:
    input:
        [directory(p) for p in OUT_DIRS]

rule checkamg_annotate:
    input:
        ptns=os.path.join(config["input_base"

    nohup snakemake \
        --use-conda \
        -j 96 \
        --snakefile ./accessory_scripts/checkamg_annotate.snakefile